# Notebook 08: Final Exploratory Data Analysis (EDA)
## Pre-IPO Subscription Forecasting for Indian Mainboard IPOs (106 Verified Records)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scipy.stats as stats

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"

# Load V2 processed datasets
features_df = pd.read_csv(PROCESSED_DIR / "ipo_features_preipo_v2.csv")
targets_df = pd.read_csv(PROCESSED_DIR / "ipo_targets_v2.csv")
merged_df = features_df.merge(targets_df, on=["ipo_id", "company_name", "ipo_open_date"])

print(f"Loaded {len(merged_df)} verified IPO observations.")
print(f"Features count: {features_df.shape[1]}, Targets count: {targets_df.shape[1]}")


### 1. Target Variables Summary Statistics


In [ ]:
target_cols = ["qib_subscription", "nii_subscription", "retail_subscription", "total_subscription"]
stats_summary = []

for t in target_cols:
    vals = merged_df[t].values
    q1, q3 = np.percentile(vals, 25), np.percentile(vals, 75)
    stats_summary.append({
        "Target": t,
        "Mean": np.mean(vals),
        "Std": np.std(vals, ddof=1),
        "Min": np.min(vals),
        "Q1 (25%)": q1,
        "Median": np.median(vals),
        "Q3 (75%)": q3,
        "Max": np.max(vals),
        "IQR": q3 - q1,
        "Skewness": stats.skew(vals),
        "Kurtosis": stats.kurtosis(vals)
    })

pd.DataFrame(stats_summary).set_index("Target")


### 2. Target Variable Empirical Cumulative Distributions (ECDF)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, t in enumerate(target_cols):
    vals = np.sort(merged_df[t].values)
    ecdf = np.arange(1, len(vals) + 1) / len(vals)
    axes[idx].plot(vals, ecdf, marker='.', linestyle='none', color='navy')
    axes[idx].set_title(f"ECDF: {t}")
    axes[idx].set_xlabel("Subscription Multiple (times)")
    axes[idx].set_ylabel("Cumulative Probability")
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 3. Sector-Wise Subscription Profiles


In [ ]:
sector_grp = merged_df.groupby("sector")[target_cols].agg(["count", "median"])
print("Sector breakdown with median subscription multiples:")
sector_grp


### 4. Correlation Analysis (Pearson & Spearman vs Targets)


In [ ]:
num_cols = [
    "issue_size_in_crore", "price_high", "pe_ratio", "debt_equity_ratio", 
    "roe_pct", "gmp_percentage", "india_vix", "previous_general_avg_subscription"
]

corr_pearson = merged_df[num_cols + target_cols].corr(method="pearson").loc[num_cols, target_cols]
corr_spearman = merged_df[num_cols + target_cols].corr(method="spearman").loc[num_cols, target_cols]

print("--- Spearman Rank Correlation with Targets ---")
corr_spearman
